In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))


# ── shared/mlfp05/ex_2.py ──
"""
Shared utilities for MLFP05 Exercise 2 — CNNs on CIFAR-10.

Common infrastructure used by all technique files:
  - CIFAR-10 data loading and normalisation
  - Device detection and precision settings
  - ExperimentTracker / ModelRegistry setup
  - LightningModule wrapper for training
  - Visualisation helpers
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pytorch_lightning as pl
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.engines.model_registry import LocalFileArtifactStore
from kailash_ml.types import MetricSpec

# ── Environment & Reproducibility ────────────────────────────────────
setup_environment()
torch.manual_seed(42)
np.random.seed(42)
pl.seed_everything(42, workers=True)

DEVICE = get_device()
# Notebook-safe path resolution: __file__ is undefined when this module is
# inlined into a Colab notebook by scripts/generate_selfcontained_notebook.py.
try:
    _HERE = Path(__file__).resolve()
    REPO_ROOT = _HERE.parents[2]
    ARTIFACT_DIR = _HERE.parent
except NameError:
    REPO_ROOT = Path.cwd()
    ARTIFACT_DIR = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"

# The ModelRegistry stores artifact files (model.pkl, model.onnx) in an
# ArtifactStore. We construct the store explicitly so exercises can attach
# an ONNX serving artifact to a registered version (see attach_onnx_artifact).
ARTIFACT_STORE = LocalFileArtifactStore(".kailash_ml/artifacts")

N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

# ── CIFAR-10 class labels (used for visualisation and apply sections) ─
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Per-channel normalisation statistics (CIFAR-10 population)
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)


# ═══════════════════════════════════════════════════════════════════════
# Data Loading
# ═══════════════════════════════════════════════════════════════════════
def load_cifar10() -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    DataLoader,
    DataLoader,
]:
    """Load and normalise the full CIFAR-10 dataset.

    Returns:
        X_train, y_train, X_val, y_val, train_loader, val_loader
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_train = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_train = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))],
        dtype=torch.long,
    )
    X_val = torch.stack([val_set[i][0] for i in range(len(val_set))])
    y_val = torch.tensor(
        [val_set[i][1] for i in range(len(val_set))],
        dtype=torch.long,
    )

    # Per-channel normalisation
    X_train = (X_train - CIFAR_MEAN) / CIFAR_STD
    X_val = (X_val - CIFAR_MEAN) / CIFAR_STD

    train_ds = TensorDataset(X_train, y_train)
    val_ds = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

    print(
        f"CIFAR-10: train {tuple(X_train.shape)}, val {tuple(X_val.shape)}, "
        f"classes={N_CLASSES}: {CLASS_NAMES}"
    )
    return X_train, y_train, X_val, y_val, train_loader, val_loader


# ═══════════════════════════════════════════════════════════════════════
# Kailash Engine Setup
# ═══════════════════════════════════════════════════════════════════════
async def setup_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Initialise ExperimentTracker (kailash-ml 1.1.1 factory) and ModelRegistry.

    Returns:
        conn, tracker, experiment_name, registry, has_registry
    """
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker and
    # ModelRegistry use incompatible _kml_model_versions schemas. Route them
    # to separate sqlite files until upstream fixes the conflict.
    db = f"sqlite:///{db_name}"
    registry_db_name = (
        db_name.replace(".db", "_registry.db")
        if db_name.endswith(".db")
        else db_name + "_registry.db"
    )
    registry_db = f"sqlite:///{registry_db_name}"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn, artifact_store=ARTIFACT_STORE)
    return conn, tracker, "m5_cnns", registry, True


def init_engines(db_name: str = "mlfp05_cnns.db") -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Sync wrapper for setup_engines."""
    return asyncio.run(setup_engines(db_name))


# ═══════════════════════════════════════════════════════════════════════
# Lightning Training Infrastructure
# ═══════════════════════════════════════════════════════════════════════
class LitCNN(pl.LightningModule):
    """Lightning wrapper for any nn.Module classifier.

    Tracks per-epoch training loss and validation accuracy for later
    logging to ExperimentTracker and visualisation with ModelVisualizer.
    """

    def __init__(self, model: nn.Module, lr: float = 1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.train_losses: list[float] = []
        self.val_accs: list[float] = []
        self._batch_losses: list[float] = []
        self._val_correct = 0
        self._val_total = 0

    def training_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        loss = F.cross_entropy(logits, y)
        self._batch_losses.append(loss.item())
        return loss

    def on_train_epoch_end(self):
        if self._batch_losses:
            self.train_losses.append(float(np.mean(self._batch_losses)))
            self._batch_losses = []

    def validation_step(self, batch, batch_idx):
        del batch_idx  # Lightning protocol arg; not used here
        x, y = batch
        logits = self.model(x)
        self._val_correct += int((logits.argmax(dim=-1) == y).sum().item())
        self._val_total += int(y.size(0))

    def on_validation_epoch_end(self):
        if self._val_total > 0:
            self.val_accs.append(self._val_correct / self._val_total)
            self._val_correct = 0
            self._val_total = 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.lr)


def get_precision_setting() -> str:
    """Return the optimal Lightning precision string for the current backend.

    Routes through ``kailash_ml.device()`` so MPS / CUDA / ROCm / Intel XPU /
    CPU are all selected by the same policy the rest of the platform uses.
    BackendInfo.precision is the canonical answer ("16-mixed" on Apple MPS
    + Tensor-Core GPUs, "32" on older GPUs and CPU).
    """
    import kailash_ml as km

    backend = km.device()
    print(f"  {backend.backend} detected -- using {backend.precision} precision")
    return backend.precision


def get_accelerator() -> str:
    """Return the Lightning accelerator string for the current backend."""
    import kailash_ml as km

    return km.device().accelerator


PRECISION = get_precision_setting()
ACCELERATOR = get_accelerator()


async def train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Train a CNN with Lightning and log metrics to ExperimentTracker.

    Uses the kailash-ml 1.1.1 ``tracker.track(...)`` async context manager.
    On normal exit the run is marked FINISHED; on exception it is marked FAILED.
    """
    lit = LitCNN(model, lr=lr)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator=ACCELERATOR,
        precision=PRECISION,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "architecture": name,
                "lr": str(lr),
                "epochs": str(epochs),
                "batch_size": str(BATCH_SIZE),
                "dataset_size": str(len(train_loader.dataset)),
                "precision": PRECISION,
                "accelerator": ACCELERATOR,
            }
        )

        trainer.fit(lit, train_loader, val_loader)

        for epoch_idx, loss in enumerate(lit.train_losses):
            await run.log_metric("train_loss", loss, step=epoch_idx + 1)
        for epoch_idx, acc in enumerate(lit.val_accs):
            await run.log_metric("val_accuracy", acc, step=epoch_idx + 1)

        await run.log_metrics(
            {
                "final_train_loss": lit.train_losses[-1],
                "final_val_accuracy": lit.val_accs[-1],
            }
        )

    print(
        f"  [{name}] final train loss={lit.train_losses[-1]:.4f}  "
        f"val acc={lit.val_accs[-1]:.3f}"
    )
    return lit.train_losses, lit.val_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    epochs: int = EPOCHS,
) -> tuple[list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        train_model_async(
            model,
            name,
            tracker,
            exp_name,
            train_loader,
            val_loader,
            lr,
            epochs,
        )
    )


# ═══════════════════════════════════════════════════════════════════════
# Model Registration
# ═══════════════════════════════════════════════════════════════════════
async def register_model_async(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Register a trained model with metrics in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="val_accuracy", value=final_acc),
            MetricSpec(name="epochs", value=float(epochs)),
            MetricSpec(name="batch_size", value=float(BATCH_SIZE)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={final_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
    final_acc: float,
    epochs: int = EPOCHS,
):
    """Sync wrapper for register_model_async."""
    return asyncio.run(
        register_model_async(registry, name, model, final_loss, final_acc, epochs)
    )


# ═══════════════════════════════════════════════════════════════════════
# Visualisation Helpers
# ═══════════════════════════════════════════════════════════════════════
def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance.

    Delegates to the canonical factory in shared.mlfp05 — the P2
    experimental notice is acknowledged there.
    """

    return _canonical()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
    x_label: str = "Epoch",
    y_label: str = "Value",
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(
        metrics=metrics,
        x_label=x_label,
        y_label=y_label,
    )
    fig.write_html(str(output_path))
    print(f"  Saved plot: {output_path}")


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters())


def denormalise_cifar(img_tensor: torch.Tensor) -> torch.Tensor:
    """Reverse CIFAR-10 normalisation for display.

    Args:
        img_tensor: shape (C, H, W) or (B, C, H, W), normalised

    Returns:
        Tensor with pixel values clipped to [0, 1]
    """
    if img_tensor.dim() == 3:
        mean = CIFAR_MEAN.squeeze(0)
        std = CIFAR_STD.squeeze(0)
    else:
        mean = CIFAR_MEAN
        std = CIFAR_STD
    return (img_tensor * std + mean).clamp(0, 1)


# ═══════════════════════════════════════════════════════════════════════
# Serving Helpers (ONNX export + InferenceServer)
# ═══════════════════════════════════════════════════════════════════════
IMAGE_SHAPE = (3, 32, 32)
N_PIXELS = 3 * 32 * 32


class FlatImageAdapter(nn.Module):
    """Wrap an image classifier so it accepts flat pixel rows.

    InferenceServer's ONNX runtime turns each request record into ONE row of
    a 2-D float array, so the exported graph must take ``(batch, 3072)``.
    This adapter reshapes rows back to ``(batch, 3, 32, 32)`` before calling
    the wrapped CNN. ``predict(X)`` is the method ``OnnxBridge.validate``
    calls to get the native (PyTorch) outputs for comparison.
    """

    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x.reshape(-1, *IMAGE_SHAPE))

    def predict(self, X) -> np.ndarray:
        device = next(self.model.parameters()).device
        with torch.no_grad():
            x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
            return self.forward(x).cpu().numpy()


def images_to_records(images: torch.Tensor) -> list[dict[str, float]]:
    """Turn a batch of images into InferenceServer request records.

    One record per image, one key per pixel. Keys are zero-padded so the
    record order is the pixel order the ONNX graph expects.
    """
    rows = images.reshape(len(images), -1).cpu().tolist()
    return [{f"px{j:04d}": float(v) for j, v in enumerate(row)} for row in rows]


def attach_onnx_artifact(name: str, version: int, onnx_path: Path) -> None:
    """Store an exported ONNX file as ``model.onnx`` for a registered version.

    ``ModelRegistry.register_model`` saves the training artifact as
    ``model.pkl``; ``InferenceServer.from_registry(..., runtime="onnx")``
    loads ``model.onnx`` from the same ArtifactStore.

    OnnxBridge writes large weights to a sibling ``<file>.onnx.data`` file
    (ONNX external data); storing only the ``.onnx`` bytes would serve a model
    without its weights, so it is stored as ONE self-contained protobuf.
    """
    import onnx

    model_proto = onnx.load(str(onnx_path))  # pulls in any .onnx.data weights
    asyncio.run(
        ARTIFACT_STORE.save(name, version, model_proto.SerializeToString(), "model.onnx")
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 2.1 — Simple CNN: Convolutions for Image Classification
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain WHY convolutions work for images (spatial hierarchy, weight
#     sharing, translation invariance) in terms a non-technical manager
#     can understand
#   - Build a CNN with Conv2d, BatchNorm2d, ReLU, and MaxPool2d layers
#   - Train on the FULL CIFAR-10 dataset (50K images) with PyTorch Lightning
#   - Track the training run with kailash-ml ExperimentTracker
#   - Visualise learned convolutional filters and feature maps to SEE what
#     the network actually learned (not just loss curves)
#   - Apply this to automated product categorisation for a Singapore
#     e-commerce platform
#
# PREREQUISITES: M5/ex_1 (autoencoders, PyTorch training basics,
#   ExperimentTracker and ModelRegistry setup)
# ESTIMATED TIME: ~35 min
#
# DATASET: CIFAR-10 — 50,000 real 32x32 colour photos across 10 classes
#   (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck).
#
# PHASES:
#   1. THEORY  — Why convolutions work for images
#   2. BUILD   — SimpleCNN architecture
#   3. TRAIN   — Train with Lightning + ExperimentTracker
#   4. VISUALISE — Learned filters + feature maps (model BEHAVIOUR)
#   5. APPLY   — Singapore e-commerce product categorisation
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt



## PHASE 1 — THEORY: Why Convolutions Work for Images


KEY CONCEPTS:
  - Local receptive fields: scan small patches, not the whole image
  - Weight sharing: same 3x3 filter applied at every position
  - Spatial hierarchy: edges -> textures -> parts -> objects
  - Translation invariance: recognise WHAT, not WHERE
  - BatchNorm: stabilise training by normalising activations
  - MaxPool: summarise spatial regions, reduce dimensions by 2x


In [ ]:
# Imagine you're a quality inspector at a factory. You don't stare at
# the entire product at once — you scan small regions looking for
# specific patterns: scratches, dents, misalignments. Then you step
# back and combine those local observations into a verdict.
#
# A convolutional neural network works the same way:
#
# 1. LOCAL RECEPTIVE FIELDS (the "scanning window"):
#    A 3x3 filter slides across the image, examining one small patch at
#    a time. A 32x32 image with a 3x3 filter produces 30x30 = 900
#    local observations per filter. This is dramatically more efficient
#    than connecting every pixel to every neuron (32x32x3 = 3,072 inputs
#    per neuron in a fully-connected layer).
#
# 2. WEIGHT SHARING (the "same inspector everywhere"):
#    The SAME 3x3 filter is applied at every spatial position. If an
#    edge detector works in the top-left corner, it works in the
#    bottom-right too. This means a 3x3 filter has only 9 learnable
#    parameters regardless of image size — a 1000x1000 image still
#    uses the same 9 weights.
#
# 3. SPATIAL HIERARCHY (the "zoom out" principle):
#    Layer 1 detects edges (horizontal, vertical, diagonal lines).
#    Layer 2 combines edges into textures (fur, fabric, metal).
#    Layer 3 combines textures into parts (wheel, wing, eye).
#    Each layer "zooms out" by one level of abstraction.
#
# 4. TRANSLATION INVARIANCE (the "it doesn't matter where"):
#    A cat in the top-left of the image activates the same filters as
#    a cat in the bottom-right. The network recognises WHAT, not WHERE.
#    MaxPool2d reinforces this by summarising each 2x2 region with its
#    maximum value.
#
# For a non-technical stakeholder: "Convolutions let the network learn
# its own inspection checklist — first small details like edges, then
# bigger patterns like shapes, then whole objects. The same checklist
# works regardless of where the object appears in the image."
#
# BatchNorm2d: After each convolution, normalise the activations so
# that training remains stable. Without it, deeper layers receive
# inputs with wildly different scales, making gradient updates erratic.
# Think of it as "recalibrating the instruments between measurements."

print("=" * 70)
print("  PHASE 1 — THEORY: Why Convolutions Work for Images")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: SimpleCNN Architecture


Architecture:
        Conv(3->32) -> BN -> ReLU -> MaxPool(2)   [32x32 -> 16x16]
        Conv(32->64) -> BN -> ReLU -> MaxPool(2)   [16x16 -> 8x8]
        Flatten -> Linear(4096->128) -> ReLU -> Linear(128->10)


In [ ]:
# Architecture:
#   Conv2d(3->32, 3x3) -> BatchNorm -> ReLU -> MaxPool (32->16)
#   Conv2d(32->64, 3x3) -> BatchNorm -> ReLU -> MaxPool (16->8)
#   Flatten -> Linear(64*8*8 -> 128) -> ReLU -> Linear(128 -> 10)
#
# Why these choices:
#   - 3x3 kernels: the smallest filter that captures spatial structure.
#     Stacking two 3x3 layers gives a 5x5 receptive field with fewer
#     parameters than a single 5x5 filter (2*9=18 vs 25).
#   - Channel doubling (32->64): deeper layers need more feature maps
#     because they represent more complex combinations of lower features.
#   - Global architecture: features (convolutions) -> head (classification).
#     This separation is standard practice — features extract, head decides.

print("\n" + "=" * 70)
print("  PHASE 2 — BUILD: SimpleCNN Architecture")
print("=" * 70)


class SimpleCNN(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 input channels (RGB) -> 32 feature maps
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # spatial: 32 -> 16
            # Block 2: 32 -> 64 feature maps
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # spatial: 16 -> 8
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))



### Checkpoint 1: Architecture builds correctly


In [ ]:
model = SimpleCNN()
dummy_input = torch.randn(2, 3, 32, 32)
dummy_output = model(dummy_input)
assert dummy_output.shape == (
    2,
    N_CLASSES,
), f"SimpleCNN output should be (batch, {N_CLASSES}), got {dummy_output.shape}"

param_count = count_parameters(model)
print(f"\nSimpleCNN built successfully")
print(f"  Parameters: {param_count:,}")
print(f"  Input shape: (batch, 3, 32, 32)")
print(f"  Output shape: (batch, {N_CLASSES})")

# INTERPRETATION: ~560K parameters is modest by modern standards (ResNet-50
# has 25M). The two conv layers learn 32+64=96 distinct 3x3 filters.
# Layer 1 filters detect basic edges; layer 2 filters detect textures
# composed from those edges. The classifier head maps the 64 feature maps
# (each 8x8) to 10 class probabilities.
print("\n--- Checkpoint 1 passed --- SimpleCNN architecture verified\n")
del model, dummy_input, dummy_output



## PHASE 3 — TRAIN: Train with Lightning + ExperimentTracker


In [ ]:
print("=" * 70)
print("  PHASE 3 — TRAIN: SimpleCNN on full CIFAR-10")
print("=" * 70)

# Load data
X_train, y_train, X_val, y_val, train_loader, val_loader = load_cifar10()



### Checkpoint 2: Data loaded correctly


In [ ]:
assert X_train.shape[0] == 50000, (
    f"Expected full 50K CIFAR-10 training set, got {X_train.shape[0]}. "
    "CNNs need the full dataset to learn general spatial features."
)
assert X_val.shape[0] == 10000, f"Expected 10K validation, got {X_val.shape[0]}"
assert X_train.shape[1:] == (3, 32, 32), "CIFAR-10 images should be 3x32x32"
print("--- Checkpoint 2 passed --- CIFAR-10 loaded (50K train, 10K val)\n")

# Set up kailash-ml engines
conn, tracker, exp_name, registry, has_registry = init_engines()

# Train
print(f"\nTraining SimpleCNN for {EPOCHS} epochs on {X_train.shape[0]:,} images...")
simple_cnn = SimpleCNN()
simple_losses, simple_accs = train_model(
    simple_cnn,
    "SimpleCNN",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)



## DIAGNOSTIC CHECKPOINT — five instruments + Grad-CAM for CNNs


Cross-entropy on one (images, labels) batch, on the model's device.


In [ ]:
# First classifier in M5: run_diagnostic_checkpoint with a cross-entropy
# loss. The probe batches come from the TRAINING loader: the probe runs
# in train mode, so BatchNorm running statistics move with whatever data
# it sees, and validation data should not leak into them.
# For CNNs, Grad-CAM is a sixth instrument — "which pixels drove this
# prediction?" — and it exposes shortcuts (Zech et al., 2018: pneumonia
# models that failed at new hospitals had learned hospital-specific
# image markers rather than pathology).
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _ce_loss(m, batch):
    xb, yb = batch
    dev = next(m.parameters()).device
    return F.cross_entropy(m(xb.to(dev)), yb.to(dev))


print("\n── Diagnostic Report (SimpleCNN) ──")
diag, findings = run_diagnostic_checkpoint(
    simple_cnn,
    train_loader,
    _ce_loss,
    title="SimpleCNN (CIFAR-10)",
    train_losses=simple_losses,
    show=False,
)
print_prescription_pad(findings, "SimpleCNN (CIFAR-10)")

# Grad-CAM on the last conv layer. grad_cam() runs on kailash-ml's
# detected device (diag.device), so the model visits it for this step.
last_conv = [n for n, mod in simple_cnn.named_modules() if isinstance(mod, nn.Conv2d)][-1]
cam_x, cam_y = next(iter(val_loader))
home_device = next(simple_cnn.parameters()).device
simple_cnn.to(diag.device)
cam = diag.grad_cam(cam_x[:4], target_class=int(cam_y[0]), layer_name=last_conv)
simple_cnn.to(home_device)
print(
    f"  Grad-CAM on '{last_conv}' for class '{CLASS_NAMES[int(cam_y[0])]}': "
    f"heatmap shape={tuple(cam.shape)}"
)
# Upsample cam[i] to 32x32 and overlay it on cam_x[i]: a hot region off
# the object means the model is leaning on background or shortcut cues.

# ══════ READING THE PRESCRIPTION PAD (key: see ex_1/01_standard_ae.py) ══════
# BatchNorm after every conv is there to keep gradient flow stable, so
# a CRITICAL gradient reading here is worth investigating before you
# trust the accuracy. Read the dead-neuron share per ReLU (weight
# sharing usually keeps conv channels active). The loss trend only
# sees training loss here; the train/val accuracy curves below show
# whether the model overfits. Never ship on accuracy alone: check the
# Grad-CAM overlays for shortcut features.



### Checkpoint 3: Training converged


In [ ]:
assert len(simple_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses"
assert simple_losses[-1] < simple_losses[0], "Loss should decrease during training"
assert simple_accs[-1] > 0.4, (
    f"SimpleCNN val accuracy {simple_accs[-1]:.3f} too low -- "
    "expected > 0.4 on full CIFAR-10 after 8 epochs"
)
# INTERPRETATION: With 50K images the SimpleCNN learns real convolutional
# filters: edge detectors in layer 1, texture detectors in layer 2. The
# accuracy above ~55% shows the model is learning meaningful spatial
# features, not just memorising. BatchNorm is crucial here -- without it,
# training would be much slower and less stable.
print(f"\nSimpleCNN final: loss={simple_losses[-1]:.4f}, val_acc={simple_accs[-1]:.3f}")
print("--- Checkpoint 3 passed --- SimpleCNN trained successfully\n")

# Register in ModelRegistry
if has_registry:

    simple_version = register_model(
        registry,
        "simple_cnn_cifar10",
        simple_cnn,
        simple_losses[-1],
        simple_accs[-1],
    )

# Save training curves
viz = create_visualizer()
save_training_plots(
    viz,
    {"SimpleCNN loss": simple_losses},
    "ex_2_01_simple_cnn_loss.html",
    y_label="Training Loss",
)
save_training_plots(
    viz,
    {"SimpleCNN val accuracy": simple_accs},
    "ex_2_01_simple_cnn_acc.html",
    y_label="Validation Accuracy",
)



## PHASE 4 — VISUALISE: Learned Filters and Feature Maps


In [ ]:
# Loss curves tell you WHETHER the model is learning. Filter and feature
# map visualisations tell you WHAT the model is learning.
#
# Layer 1 filters should look like oriented edge detectors (Gabor-like
# patterns). Feature maps show which parts of the input image activate
# each filter — bright regions = strong activation = "this filter found
# something here."

print("=" * 70)
print("  PHASE 4 — VISUALISE: What Did the CNN Learn?")
print("=" * 70)

# 4a. Visualise Layer 1 learned filters (3x3 RGB filters)
layer1_weights = simple_cnn.features[0].weight.data.cpu()  # (32, 3, 3, 3)

fig_filters, axes = plt.subplots(4, 8, figsize=(16, 8))
fig_filters.suptitle(
    "SimpleCNN Layer 1 Learned Filters (32 filters, 3x3 RGB)", fontsize=14
)
for i, ax in enumerate(axes.flat):
    if i < layer1_weights.shape[0]:
        # Normalise each filter to [0, 1] for display
        filt = layer1_weights[i]  # (3, 3, 3) — RGB
        filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
        ax.imshow(filt.permute(1, 2, 0).numpy())  # (3,3,3) -> (H,W,C)
    ax.axis("off")
plt.tight_layout()
plt.savefig("ex_2_01_learned_filters.png", dpi=150, bbox_inches="tight")
plt.close(fig_filters)
print("  Saved: ex_2_01_learned_filters.png")
print("  Look for: edge detectors (horizontal, vertical, diagonal lines)")
print("  Some filters will be colour-sensitive (responding to red/green/blue edges)")

# 4b. Visualise feature maps for a sample image
simple_cnn.eval()
sample_idx = 7  # A horse in CIFAR-10
sample_img = X_val[sample_idx : sample_idx + 1]  # (1, 3, 32, 32)
true_label = CLASS_NAMES[y_val[sample_idx].item()]

# Extract intermediate feature maps using hooks
feature_maps = {}


def hook_fn(name):
    def hook(module, input, output):
        feature_maps[name] = output.detach().cpu()

    return hook


handles = []
handles.append(simple_cnn.features[0].register_forward_hook(hook_fn("conv1_raw")))
handles.append(simple_cnn.features[2].register_forward_hook(hook_fn("conv1_relu")))
handles.append(simple_cnn.features[4].register_forward_hook(hook_fn("conv2_raw")))
handles.append(simple_cnn.features[6].register_forward_hook(hook_fn("conv2_relu")))

with torch.no_grad():
    pred_logits = simple_cnn(sample_img)
    pred_class = CLASS_NAMES[pred_logits.argmax(dim=-1).item()]

for h in handles:
    h.remove()

# Plot: original image + layer 1 feature maps + layer 2 feature maps
fig_fmaps, axes = plt.subplots(3, 9, figsize=(18, 7))
fig_fmaps.suptitle(
    f"Feature Maps for '{true_label}' (predicted: '{pred_class}')",
    fontsize=14,
)

# Row 0: original image (centre) + first 8 conv1 ReLU feature maps
orig_display = denormalise_cifar(sample_img.squeeze(0))
axes[0, 0].imshow(orig_display.permute(1, 2, 0).numpy())
axes[0, 0].set_title("Original", fontsize=9)
axes[0, 0].axis("off")
conv1_maps = feature_maps["conv1_relu"].squeeze(0)  # (32, 32, 32)
for i in range(8):
    axes[0, i + 1].imshow(conv1_maps[i].numpy(), cmap="viridis")
    axes[0, i + 1].set_title(f"L1 F{i}", fontsize=8)
    axes[0, i + 1].axis("off")

# Row 1: next 8 conv1 feature maps
for i in range(8):
    idx = i + 8
    if idx < conv1_maps.shape[0]:
        axes[1, i].imshow(conv1_maps[idx].numpy(), cmap="viridis")
        axes[1, i].set_title(f"L1 F{idx}", fontsize=8)
    axes[1, i].axis("off")
axes[1, 8].axis("off")

# Row 2: 8 conv2 ReLU feature maps (16x16, captured before the second MaxPool)
conv2_maps = feature_maps["conv2_relu"].squeeze(0)  # (64, 16, 16)
for i in range(8):
    axes[2, i].imshow(conv2_maps[i].numpy(), cmap="magma")
    axes[2, i].set_title(f"L2 F{i}", fontsize=8)
    axes[2, i].axis("off")
axes[2, 8].axis("off")

plt.tight_layout()
plt.savefig("ex_2_01_feature_maps.png", dpi=150, bbox_inches="tight")
plt.close(fig_fmaps)
print(f"\n  Saved: ex_2_01_feature_maps.png")
print(f"  Sample: true='{true_label}', predicted='{pred_class}'")
print("  Layer 1 maps: edges and colour boundaries (high-res, 32x32)")
print("  Layer 2 maps: textures and shapes (lower-res, 8x8 after pooling)")

# 4c. Visualise prediction confidence on a batch of images
with torch.no_grad():
    batch_imgs = X_val[:16]
    batch_logits = simple_cnn(batch_imgs)
    batch_probs = F.softmax(batch_logits, dim=-1)
    batch_preds = batch_logits.argmax(dim=-1)

fig_preds, axes = plt.subplots(2, 8, figsize=(20, 6))
fig_preds.suptitle("SimpleCNN Predictions on Validation Images", fontsize=14)
for i in range(16):
    row, col = i // 8, i % 8
    img_display = denormalise_cifar(batch_imgs[i])
    axes[row, col].imshow(img_display.permute(1, 2, 0).numpy())
    pred_name = CLASS_NAMES[batch_preds[i].item()]
    true_name = CLASS_NAMES[y_val[i].item()]
    conf = batch_probs[i][batch_preds[i]].item()
    correct = batch_preds[i].item() == y_val[i].item()
    colour = "green" if correct else "red"
    axes[row, col].set_title(
        f"{pred_name}\n({conf:.0%})",
        fontsize=8,
        color=colour,
    )
    axes[row, col].axis("off")
plt.tight_layout()
plt.savefig("ex_2_01_predictions.png", dpi=150, bbox_inches="tight")
plt.close(fig_preds)
print("  Saved: ex_2_01_predictions.png")
print("  Green = correct, Red = incorrect. Confidence shown in parentheses.")



### Checkpoint 4: Visualisations generated


In [ ]:
import os

assert os.path.exists("ex_2_01_learned_filters.png"), "Filter visualisation missing"
assert os.path.exists("ex_2_01_feature_maps.png"), "Feature map visualisation missing"
assert os.path.exists("ex_2_01_predictions.png"), "Prediction visualisation missing"
print("\n--- Checkpoint 4 passed --- visual proof of model behaviour saved\n")



## PHASE 5 — APPLY: Singapore E-Commerce Product Categorisation


PRODUCTION TRIAGE SIMULATION (10,000 product listings):

  Category         | Count  | % Total | Accuracy
  -----------------+--------+---------+---------
  Auto-approved    | {n_auto:>5,} | {n_auto/n_total:>6.1%}  | {auto_acc:.1%}
  Needs review     | {n_review:>5,} | {n_review/n_total:>6.1%}  | {review_acc:.1%}
  Low confidence   | {n_reject:>5,} | {n_reject/n_total:>6.1%}  | (routed to human)

  BUSINESS IMPACT (daily projection for 500K listings):
    Auto-approved listings:   {int(500000 * n_auto/n_total):>7,} (no human cost)
    Human review needed:      {int(500000 * n_review/n_total):>7,} (@ $0.02/review)
    Daily review cost:        ${int(500000 * (n_review + n_reject)/n_total * 0.02):>7,}
    vs. full manual review:   $  10,000
    Daily savings:            ${10000 - int(500000 * (n_review + n_reject)/n_total * 0.02):>7,}
    Projected annual savings: ${(10000 - int(500000 * (n_review + n_reject)/n_total * 0.02)) * 365:>10,}

  STAKEHOLDER INSIGHT:
    Auto-approved accuracy of {auto_acc:.1%} means {100 - auto_acc*100:.1f}% error rate
    on automated decisions. For an e-commerce platform, this means
    roughly {int(500000 * n_auto/n_total * (1 - auto_acc)):,} mis-categorised products
    per day slip through without human review.

    RECOMMENDATION: Accept this if mis-categorisation cost < $0.50/item.
    For high-value categories (electronics, luxury), lower the auto-approve
    threshold to 0.95 and accept higher review volume.


In [ ]:
# SCENARIO: You are an ML engineer at a Singapore e-commerce platform
# (illustrative figures). The platform receives 500,000+
# new product listings per day. Sellers often mis-categorise products
# (a pair of running shoes listed under "Electronics" instead of "Shoes"),
# leading to:
#   - Poor search results (customers can't find what they want)
#   - Incorrect commission rates (different categories have different fees)
#   - Regulatory compliance issues (restricted items in wrong categories)
#
# BUSINESS CASE:
#   - Manual review: 500K listings/day * $0.02/listing = $10,000/day
#   - CNN auto-categorisation: $500/day compute + $2,000/day for edge cases
#   - Net savings: ~$7,500/day = ~$2.7M/year
#   - Additional revenue: improved search -> +3-5% conversion rate
#
# HOW THE CNN APPLIES:
#   Our SimpleCNN learned to classify 32x32 images into 10 categories.
#   A production system would:
#   1. Resize product images to standard dimensions
#   2. Run through a CNN trained on product categories (not CIFAR-10)
#   3. High-confidence predictions (>0.9) auto-categorise
#   4. Low-confidence predictions route to human reviewers
#   5. Track accuracy with ExperimentTracker, retrain monthly
#
# CIFAR-10 AS PROXY:
#   CIFAR-10's categories (airplane, automobile, truck, ship, etc.) map
#   to real e-commerce categories. The architecture patterns are
#   identical — only the training data changes.

print("=" * 70)
print("  PHASE 5 — APPLY: Singapore E-Commerce Product Categorisation")
print("=" * 70)

# Simulate the production triage system using our trained SimpleCNN
simple_cnn.eval()
HIGH_CONFIDENCE_THRESHOLD = 0.85
LOW_CONFIDENCE_THRESHOLD = 0.50

# Map CIFAR-10 classes to e-commerce product categories
ECOMMERCE_MAPPING = {
    "airplane": "Travel & Luggage",
    "automobile": "Automotive",
    "bird": "Pet Supplies",
    "cat": "Pet Supplies",
    "deer": "Home & Garden (Decor)",
    "dog": "Pet Supplies",
    "frog": "Toys & Collectibles",
    "horse": "Sports & Outdoors",
    "ship": "Travel & Luggage",
    "truck": "Automotive",
}

with torch.no_grad():
    val_logits = simple_cnn(X_val)
    val_probs = F.softmax(val_logits, dim=-1)
    val_preds = val_logits.argmax(dim=-1)
    val_confidences = val_probs.gather(1, val_preds.unsqueeze(1)).squeeze()
    val_correct = (val_preds == y_val).float()

# Triage: auto-approve, review, reject
auto_approve_mask = val_confidences >= HIGH_CONFIDENCE_THRESHOLD
review_mask = (val_confidences >= LOW_CONFIDENCE_THRESHOLD) & ~auto_approve_mask
reject_mask = val_confidences < LOW_CONFIDENCE_THRESHOLD

n_total = len(y_val)
n_auto = auto_approve_mask.sum().item()
n_review = review_mask.sum().item()
n_reject = reject_mask.sum().item()

auto_acc = val_correct[auto_approve_mask].mean().item() if n_auto > 0 else 0
review_acc = val_correct[review_mask].mean().item() if n_review > 0 else 0

print(
)

# Show example auto-approved and review-needed listings
print("  EXAMPLE AUTO-APPROVED LISTINGS:")
auto_indices = torch.where(auto_approve_mask)[0][:5]
for idx in auto_indices:
    pred_name = CLASS_NAMES[val_preds[idx].item()]
    true_name = CLASS_NAMES[y_val[idx].item()]
    conf = val_confidences[idx].item()
    ecom_cat = ECOMMERCE_MAPPING[pred_name]
    status = "CORRECT" if pred_name == true_name else "MIS-CATEGORISED"
    print(
        f"    Listing #{idx.item()}: {pred_name} -> {ecom_cat} "
        f"(conf={conf:.2f}) [{status}]"
    )

print("\n  EXAMPLE REVIEW-NEEDED LISTINGS:")
review_indices = torch.where(review_mask)[0][:5]
for idx in review_indices:
    pred_name = CLASS_NAMES[val_preds[idx].item()]
    true_name = CLASS_NAMES[y_val[idx].item()]
    conf = val_confidences[idx].item()
    print(
        f"    Listing #{idx.item()}: predicted={pred_name} (conf={conf:.2f}), "
        f"true={true_name} -> NEEDS HUMAN REVIEW"
    )



### Checkpoint 5: Apply section complete


In [ ]:
assert n_auto + n_review + n_reject == n_total, "Triage should cover all samples"
assert auto_acc > 0.7, (
    f"Auto-approved accuracy {auto_acc:.3f} too low -- "
    "high-confidence predictions should be mostly correct"
)
print("\n--- Checkpoint 5 passed --- e-commerce application demonstrated\n")



## Clean up


In [ ]:

await conn.close()



## REFLECTION


THEORY:
  [x] Convolutions scan local patches with shared weights -- 3x3 filters
      detect edges at every position with only 9 parameters
  [x] Spatial hierarchy: edges -> textures -> parts -> objects, each
      layer building on the previous
  [x] BatchNorm stabilises training; MaxPool adds translation invariance

  BUILD + TRAIN:
  [x] SimpleCNN: Conv2d + BatchNorm + ReLU + MaxPool, two-block design
  [x] Trained on FULL CIFAR-10 (50K images) with ExperimentTracker
  [x] ~{param_count:,} parameters, achieves >{simple_accs[-1]:.0%} validation accuracy

  VISUALISE (the proof):
  [x] Layer 1 filters: oriented edge detectors (Gabor-like patterns)
  [x] Feature maps: which spatial regions activate each filter
  [x] Prediction grid: where the model succeeds and fails, with confidence

  APPLY:
  [x] Singapore e-commerce product auto-categorisation
  [x] Confidence-based triage: auto-approve vs human review
  [x] Business impact: projected annual savings from automation
  [x] Stakeholder-ready accuracy and error rate analysis

  KEY INSIGHT: A CNN is not a black box. The filters it learns are
  interpretable -- they are the visual features the network considers
  important. Filter and feature map visualisation is how you debug and
  explain CNN decisions to non-technical stakeholders.

  Next: In 02_resnet_se.py, you'll see why deeper networks fail and how
  residual connections and attention mechanisms fix the problem...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

